# BICE RAG — Notebook de Exploración
Prueba interactiva de chunking, embeddings y retriever.
Ejecutar con el backend corriendo: `docker-compose up backend`

## 0. Setup

In [ ]:
# Instalar dependencias si no están
%pip install langchain langchain-ollama langchain-chroma pypdf pandas matplotlib --quiet

In [ ]:
import sys
import os

# Apuntar al backend para usar los módulos del proyecto
sys.path.insert(0, '../backend')

# Verificar que Ollama está corriendo
import requests
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=3)
    models = [m['name'] for m in r.json().get('models', [])]
    print('✅ Ollama corriendo. Modelos disponibles:')
    for m in models:
        print(f'   - {m}')
except Exception as e:
    print(f'❌ Ollama no responde: {e}')
    print('   Ejecuta: ollama serve')

## 1. Exploración de Chunking
Compara cómo distintas estrategias parten los documentos.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Cargar un documento de prueba
DOC_PATH = '../backend/data/docs/auto_poliza_basica.txt'
loader = TextLoader(DOC_PATH, encoding='utf-8')
docs = loader.load()

print(f'Documento cargado: {len(docs[0].page_content)} caracteres')
print('---')
print(docs[0].page_content[:300])

In [ ]:
import pandas as pd

# Probar distintos tamaños de chunk
configs = [
    {'chunk_size': 200, 'chunk_overlap': 40},
    {'chunk_size': 500, 'chunk_overlap': 100},  # config actual del proyecto
    {'chunk_size': 800, 'chunk_overlap': 160},
]

results = []
for cfg in configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg['chunk_size'],
        chunk_overlap=cfg['chunk_overlap']
    )
    chunks = splitter.split_documents(docs)
    sizes = [len(c.page_content) for c in chunks]
    results.append({
        'chunk_size': cfg['chunk_size'],
        'overlap': cfg['chunk_overlap'],
        'n_chunks': len(chunks),
        'avg_chars': round(sum(sizes)/len(sizes)),
        'min_chars': min(sizes),
        'max_chars': max(sizes)
    })

df = pd.DataFrame(results)
print('Comparación de estrategias de chunking:')
display(df)

In [ ]:
# Ver un chunk específico — ¿parte cláusulas legales?
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(docs)

print(f'Total chunks con config actual (500/100): {len(chunks)}')
print('\n--- Chunk 0 ---')
print(chunks[0].page_content)
print('\n--- Chunk 1 ---')
print(chunks[1].page_content)
print('\n--- Último chunk ---')
print(chunks[-1].page_content)

## 2. Exploración de Embeddings
Verifica que los embeddings se generan correctamente con nomic-embed-text.

In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model='nomic-embed-text')

# Generar embedding de prueba
test_text = 'cobertura de robo del vehículo'
vector = embeddings.embed_query(test_text)

print(f'✅ Embedding generado correctamente')
print(f'   Dimensiones: {len(vector)}')
print(f'   Primeros 5 valores: {[round(v, 4) for v in vector[:5]]}')

In [ ]:
import numpy as np

# Medir similitud coseno entre queries relacionadas y no relacionadas
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

queries = [
    ('¿Qué cubre el seguro de auto?', '¿Cuál es la cobertura automotriz?'),      # muy similares
    ('¿Qué cubre el seguro de auto?', '¿Cuánto vale una acción de Apple?'),       # no relacionadas
    ('deducible de incendio', 'monto a pagar en caso de fuego'),                  # semánticamente similares
]

print('Similitud coseno entre queries:')
print('-' * 60)
for q1, q2 in queries:
    v1 = embeddings.embed_query(q1)
    v2 = embeddings.embed_query(q2)
    sim = cosine_similarity(v1, v2)
    print(f'  {sim:.4f} | "{q1[:35]}" vs "{q2[:35]}"')

## 3. Exploración del Retriever
Prueba qué chunks recupera el sistema y con qué scores.

In [ ]:
from langchain_chroma import Chroma

# Cargar el ChromaDB ya indexado por el backend
CHROMA_PATH = '../backend/chroma_db'

try:
    db = Chroma(
        persist_directory=CHROMA_PATH,
        embedding_function=embeddings
    )
    count = db._collection.count()
    print(f'✅ ChromaDB cargado: {count} chunks indexados')
except Exception as e:
    print(f'❌ Error cargando ChromaDB: {e}')
    print('   Ejecuta primero: docker exec -it bice-backend python -m src.ingestion.ingest')

In [ ]:
# Probar retrieval con scores — la clave para entender el retrieval guard
def retrieve_with_scores(query: str, k: int = 6):
    results = db.similarity_search_with_score(query, k=k)
    print(f'Query: "{query}"')
    print(f'SCORE_THRESHOLD actual: 1.2 (menor = más relevante en ChromaDB)')
    print('-' * 70)
    for i, (doc, score) in enumerate(results):
        relevante = '✅' if score <= 1.2 else '❌ BLOQUEADO'
        fuente = doc.metadata.get('source', 'desconocido')
        print(f'[{i+1}] Score: {score:.4f} {relevante}')
        print(f'     Fuente: {fuente}')
        print(f'     Texto: {doc.page_content[:120]}...')
        print()

# Pregunta dentro del dominio
retrieve_with_scores('¿Qué cubre el seguro de auto en caso de robo?')

In [ ]:
# Pregunta fuera del dominio — debe ser bloqueada por el retrieval guard
retrieve_with_scores('¿Cuánto vale una acción de Apple hoy?')

In [ ]:
# Prueba con filtro de metadata por tipo de seguro
results = db.similarity_search_with_score(
    '¿Cuál es el deducible?',
    k=4,
    filter={'tipo_seguro': 'auto'}
)
print('Retrieval filtrado por tipo_seguro=auto:')
for doc, score in results:
    print(f'  Score: {score:.4f} | Fuente: {doc.metadata.get("source")}')
    print(f'  {doc.page_content[:100]}...')
    print()

## 4. Exploración end-to-end
Prueba la chain completa llamando directamente al módulo del backend.

In [ ]:
from src.chain.rag_chain import ask

# Pregunta normal
result = ask('¿Qué cubre el seguro de auto en caso de incendio?', session_id='notebook-test')
print('RESPUESTA:')
print(result['answer'])
print(f'\nFuentes usadas: {result["sources"]}')
print(f'Chunks usados: {result["chunks_used"]}')

In [ ]:
# Prueba de memoria conversacional
SESSION = 'notebook-memoria-test'

r1 = ask('¿Qué es el deducible?', session_id=SESSION)
print('Turno 1:', r1['answer'][:200])

r2 = ask('¿Y cuánto es en el seguro de hogar?', session_id=SESSION)  # referencia al contexto anterior
print('\nTurno 2 (debería recordar contexto):', r2['answer'][:200])

In [ ]:
# Prueba del retrieval guard
r = ask('¿Cuánto vale el dólar hoy?', session_id='notebook-guard-test')
print('Retrieval guard:', r['answer'])
print('Chunks usados:', r['chunks_used'])  # debe ser 0